# 1. Information about the submission

## 1.1 Name and number of the assignment

Hallucination Detection in Tool Calling

## 1.2 Student name

Erik Shaikhiev

Tishchenko Margarita

Artemii Rubtcov

Roman Branovets

Andrej Mymrin

## 1.3 Codalab user ID / nickname / username

Not applicable for this local reproducibility notebook.

## 1.4 Additional comments

The notebook is written to be reproducible both inside the local repository and in a clean Colab-like environment. It uses the public GitHub repository `Eroouu/transformers_project.git` and the prepared ToolACE-derived datasets when they are available.

# 2. Technical Report

*Use Section 2 to describe results of your experiments as you would do writing a paper about your results. DO NOT insert code in this part. Only insert plots and tables summarizing results as needed. Use formulas if needed do described your methodology. The code is provided in Section 3.*

## 2.1 Methodology

The assignment asks us to detect span-level hallucinations in tool-calling dialogues. Following the provided task description, each example is represented in a RAGTruth-style schema: `query` is the user question, `context` is the tool output, `output` is the final model answer, and `hallucination_labels` contains character-level spans that mark unsupported text. The project builds on ToolACE dialogues and creates three corrupted subsets: contradiction with the tool output, overgeneration beyond the tool output, and missing-tool suggestions that require unavailable tools. Clean examples are kept as negative cases.

The repository implements the full pipeline in Python. Dataset construction and validation live in `data/`, while training and evaluation live in `src/`. The main improved model is a LettuceDetect-compatible token classifier fine-tuned on the generated span labels. During preprocessing, the tool output and user query are formatted as context/question, the final answer is tokenized as the second sequence, and only answer tokens receive binary labels: `supported` or `hallucination`. To handle class imbalance, the trainer can use inverse-frequency class weights.

For a reproducible run, the notebook first locates or clones `https://github.com/Eroouu/transformers_project.git`, installs the project dependencies, validates the final dataset split, trains the token classifier on `final_dataset_train`, and evaluates on `final_dataset_test`. The default configuration uses a short `max_steps` smoke training run so that the notebook can execute on limited hardware; setting `QUICK_RUN = False` switches to the full training schedule. Metrics are span-level Precision, Recall, and F1 computed by the repository evaluator through overlap between predicted and gold hallucination spans.

## 2.2 Discussion of results

The final dataset contains four files: `clean.jsonl`, `contradiction.jsonl`, `overgeneration.jsonl`, and `missing_tool.jsonl`. In the local prepared split, each corruption type has 2431 examples in total, with 1945 train and 486 test examples; the clean split has the same number of negative examples. This gives 7779 train examples and 1944 held-out test examples across all four files.

The heuristic `tool_overlap` baseline has very high recall but low precision because it flags many answer tokens that are simply absent from the raw tool output. On the local held-out test split it gives `TP=2099`, `FP=13582`, `FN=18`, `P=0.1339`, `R=0.9915`, `F1=0.2359`. The fine-tuned LettuceDetect-compatible model is the main improved method: the repository README records the latest local full-run result on the ToolACE test split as `P=0.9614`, `R=0.9719`, `F1=0.9666`, while LookBack Lens reached `P=0.4185`, `R=0.9756`, `F1=0.5858`. The code below recomputes the metrics for the current run and prints a comparison table.

# 3. Code

*Enter here all code used to produce your results submitted to Codalab. Add some comments and subsections to navigate though your solution.*

*In this part you are expected to develop yourself a solution of the task and provide a reproducible code:*
- *Using Python 3;*
- *Contains code for installation of all dependencies;*
- *Contains code for downloading of all the datasets used*;
- *Contains the code for reproducing your results (in other words, if a tester downloads your notebook she should be able to run cell-by-cell the code and obtain your experimental results as described in the methodology section)*.


*As a result, you code will be graded according to these criteria:*
- ***Readability**: your code should be well-structured preferably with indicated parts of your approach (Preprocessing, Model training, Evaluation, etc.).*
- ***Reproducibility**: your code should be reproduced without any mistakes with “Run all” mode (obtaining experimental part).*


## 3.1 Requirements

In [1]:
# 3.1 Requirements and repository setup
# This cell works both inside the local repository and in a clean notebook runtime.

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Eroouu/transformers_project.git"
REPO_DIR = Path("transformers_project")

cwd = Path.cwd()
if (cwd / "src" / "train_lettucedetect.py").exists() and (cwd / "requirements.txt").exists():
    project_dir = cwd
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    project_dir = REPO_DIR.resolve()
    os.chdir(project_dir)

print(f"Project directory: {Path.cwd()}")
print(f"Repository URL: {REPO_URL}")

INSTALL_DEPS = True
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


Project directory: /content/transformers_project
Repository URL: https://github.com/Eroouu/transformers_project.git


## 3.2 Download the data

### Dataset source and construction

The experiments use the prepared dataset files that are now stored directly in the repository after the latest pull. The starting point is ToolACE-style tool-calling data: each record contains a user request, the tool output available to the assistant, and the assistant's final answer. We convert these records into a RAGTruth-style span-detection format with four fields that matter for the models: `query`, `context`, `output`, and `hallucination_labels`.

The final dataset is organized into four aligned JSONL files:

- `clean.jsonl`: the original answer is kept and `hallucination_labels` is empty.
- `contradiction.jsonl`: a factual value from the answer is changed so it conflicts with the tool output.
- `overgeneration.jsonl`: an unsupported but plausible claim is inserted into the answer.
- `missing_tool.jsonl`: the answer claims an action was completed even though the required tool call is absent.

The repository scripts under `data/` are responsible for this construction. In particular, `data/generate_hallucinations.py` creates corrupted examples and validates character-level span offsets, while `data/merge_datasets.py` prepares the final train/test folders used below. In this notebook we first check that those prepared files are present, then inspect several examples before training or evaluating any model.


In [2]:
# 3.2 Download / locate the data
# The latest repository version already contains prepared final datasets.
# In a fresh Colab runtime this cell also works after cloning the repository in Section 3.1.

from pathlib import Path

DATASET_DIR = Path("final_dataset")
TRAIN_DIR = Path("final_dataset_train")
TEST_DIR = Path("final_dataset_test")
DATASET_FILES = ["clean.jsonl", "contradiction.jsonl", "overgeneration.jsonl", "missing_tool.jsonl"]

for dataset_dir in [DATASET_DIR, TRAIN_DIR, TEST_DIR]:
    missing = [name for name in DATASET_FILES if not (dataset_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing files in {dataset_dir}: {missing}. "
            "Run data/generate_hallucinations.py and data/merge_datasets.py, "
            "or pull the prepared dataset artifacts from the repository."
        )

print("Found prepared datasets:")
for dataset_dir in [DATASET_DIR, TRAIN_DIR, TEST_DIR]:
    print(f"- {dataset_dir.resolve()}")


Found prepared datasets:
- /content/transformers_project/final_dataset
- /content/transformers_project/final_dataset_train
- /content/transformers_project/final_dataset_test


### What one record looks like

Each JSONL line is one training or evaluation example. The `hallucination_labels` list stores character spans inside `output`, so the task is not only to decide whether the answer is wrong, but also to mark exactly which text is unsupported. The next cell prints one compact example from each corruption type and checks that every annotated span points to the same text that is stored in the label.


In [3]:
# Show compact, readable examples from all four dataset files.

import json
import textwrap
from pathlib import Path


def load_first(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.loads(next(f))


def shorten(text: str, width: int = 420) -> str:
    text = " ".join(str(text).split())
    return textwrap.shorten(text, width=width, placeholder=" ...")


def show_example(example: dict, title: str) -> None:
    print("=" * 100)
    print(title)
    print("-" * 100)
    print("QUERY:")
    print(shorten(example.get("query", "")))
    print("\nTOOL CONTEXT:")
    print(shorten(example.get("context", "")))
    print("\nMODEL OUTPUT:")
    print(shorten(example.get("output", "")))
    labels = example.get("hallucination_labels", [])
    print(f"\nANNOTATED SPANS: {len(labels)}")
    if not labels:
        print("No hallucination span: this is a clean example.")
    for label in labels[:3]:
        start, end = int(label["start"]), int(label["end"])
        output_text = example.get("output", "")
        span_text = output_text[start:end]
        print({
            "type": label.get("type"),
            "start": start,
            "end": end,
            "label_text": label.get("text"),
            "text_from_output_offsets": span_text,
            "offsets_match_label_text": span_text == label.get("text"),
        })
    print()

for filename in DATASET_FILES:
    example = load_first(TEST_DIR / filename)
    show_example(example, title=f"{filename} / corruption_type={example.get('corruption_type')}")


clean.jsonl / corruption_type=clean
----------------------------------------------------------------------------------------------------
QUERY:
Could you please find me some quotes about "inspiration"?

TOOL CONTEXT:
[{"name": "Quotes by Keywords", "results": {"quotes": [{"text": "The only way to achieve the impossible is to believe it is possible.", "author": "Charles Kingsleigh"}, {"text": "Don't watch the clock; do what it does. Keep going.", "author": "Sam Levenson"}, {"text": "Success is not the key to happiness. Happiness is the key to success. If you love what you are doing, you will be successful.", "author": "Albert ...

MODEL OUTPUT:
Here are some inspiration quotes for you: 1. "The only way to achieve the impossible is to believe it is possible." - Charles Kingsleigh 2. "Don't watch the clock; do what it does. Keep going." - Sam Levenson 3. "Success is not the key to happiness. Happiness is the key to success. If you love what you are doing, you will be successful." - Albert

## 3.3 Preprocessing

### Preprocessing and validation logic

The model code consumes the JSONL files directly, so preprocessing here is intentionally lightweight. We count the number of examples and gold spans in each split, compute an average span length for a sanity check, and run the repository validator. This matters because a span-level hallucination detector is very sensitive to off-by-one errors: if `start` and `end` do not point to the exact substring in `output`, both training and evaluation become misleading.


In [4]:
# 3.3 Preprocessing and validation
# Count examples and spans, then run the repository validator over the final dataset.
# The train/test folders are already split: train is used for fitting models, test is held out.

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd


def summarize_dataset(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for name in DATASET_FILES:
        path = dataset_dir / name
        examples = 0
        examples_with_labels = 0
        spans = 0
        span_chars = 0
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                labels = item.get("hallucination_labels", [])
                examples += 1
                examples_with_labels += int(bool(labels))
                spans += len(labels)
                span_chars += sum(int(label["end"]) - int(label["start"]) for label in labels)
        rows.append({
            "split": dataset_dir.name,
            "file": name,
            "examples": examples,
            "examples_with_labels": examples_with_labels,
            "gold_spans": spans,
            "avg_span_chars": round(span_chars / spans, 2) if spans else 0.0,
        })
    return pd.DataFrame(rows)

stats = pd.concat(
    [summarize_dataset(DATASET_DIR), summarize_dataset(TRAIN_DIR), summarize_dataset(TEST_DIR)],
    ignore_index=True,
)
display(stats)

subprocess.run([sys.executable, "data/validate_corrupted_datasets.py", str(DATASET_DIR)], check=True)


,split,file,examples,examples_with_labels,gold_spans,avg_span_chars
0,final_dataset,clean.jsonl,2431,0,0,0.00
1,final_dataset,contradiction.jsonl,2431,2431,2431,16.28
2,final_dataset,overgeneration.jsonl,2431,2431,2431,66.61
3,final_dataset,missing_tool.jsonl,2431,2431,2431,63.06
4,final_dataset_train,clean.jsonl,1945,0,0,0.00
5,final_dataset_train,contradiction.jsonl,1945,1945,1945,16.08
6,final_dataset_train,overgeneration.jsonl,1945,1945,1945,66.75
7,final_dataset_train,missing_tool.jsonl,1945,1945,1945,62.94
8,final_dataset_test,clean.jsonl,486,0,0,0.00
9,final_dataset_test,contradiction.jsonl,486,486,486,17.11


CompletedProcess(args=['/usr/bin/python3', 'data/validate_corrupted_datasets.py', 'final_dataset'], returncode=0)

## 3.4 My method of text processing

### Training and evaluation plan

This section trains the LettuceDetect-compatible model and evaluates the baselines. The previous version failed at `lookback_lens` because the notebook only checked whether `models/lookback_lens` existed; a directory can exist even when the classifier inside it is missing, stale, or incompatible. The subprocess call also used `capture_output=True` without printing `stderr`, so Colab showed only a generic `CalledProcessError` instead of the real traceback.

The cells below fix that in two ways. First, every subprocess is executed through a helper that prints both `stdout` and `stderr` if something fails. Second, LookBack Lens gets its own training cell and writes to a notebook-specific directory, so it does not accidentally reuse an old classifier.


In [ ]:
# 3.4 Training setup and shared helpers
# QUICK_RUN keeps the notebook practical for Colab smoke tests. Set QUICK_RUN = False
# for the full experiment used in the report.

import json
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch

QUICK_RUN = True
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LETTUCE_BASE_MODEL = "KRLabsOrg/lettucedect-base-modernbert-en-v1"
LETTUCE_OUTPUT_DIR = Path("models/assignment_lettucedetect_quick" if QUICK_RUN else "models/assignment_lettucedetect_full")

LOOKBACK_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
LOOKBACK_DIR = Path("models/assignment_lookback_lens_quick" if QUICK_RUN else "models/assignment_lookback_lens_full")
LOOKBACK_LIMIT = 96 if QUICK_RUN else None
LOOKBACK_MAX_LENGTH = 512 if QUICK_RUN else 2048
LOOKBACK_EVAL_LIMIT_PER_FILE = 24 if QUICK_RUN else None


def run_checked(cmd: list[str]) -> subprocess.CompletedProcess:
    print("$ " + " ".join(map(str, cmd)))
    completed = subprocess.run(cmd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print("STDERR:")
            print(completed.stderr)
        raise subprocess.CalledProcessError(
            completed.returncode,
            completed.args,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    if completed.stderr:
        print("STDERR:")
        print(completed.stderr)
    return completed


def make_eval_subset(source_dir: Path, output_path: Path, limit_per_file: int | None) -> Path:
    """Create a small mixed JSONL evaluation file for quick neural-baseline comparison."""
    if limit_per_file is None:
        return source_dir
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as out_f:
        for name in DATASET_FILES:
            with (source_dir / name).open("r", encoding="utf-8") as in_f:
                for idx, line in enumerate(in_f):
                    if idx >= limit_per_file:
                        break
                    out_f.write(line)
    return output_path


COMPARISON_DATASET = make_eval_subset(
    TEST_DIR,
    Path("outputs") / "notebook_quick_comparison_test.jsonl",
    LOOKBACK_EVAL_LIMIT_PER_FILE,
)
print(f"Device: {DEVICE}")
print(f"Comparison dataset: {COMPARISON_DATASET}")


### Train LettuceDetect-compatible detector

The LettuceDetect-compatible model is the main fine-tuned detector. In `QUICK_RUN` mode we train only for a few optimizer steps to keep the notebook runnable in Colab; the resulting metrics are a smoke test, not the full report score.


In [ ]:
# Train LettuceDetect-compatible model.

lettuce_train_cmd = [
    sys.executable,
    "src/train_lettucedetect.py",
    "--dataset", str(TRAIN_DIR),
    "--output_dir", str(LETTUCE_OUTPUT_DIR),
    "--model", LETTUCE_BASE_MODEL,
    "--device", DEVICE,
    "--epochs", "1" if QUICK_RUN else "3",
    "--max_steps", "20" if QUICK_RUN else "-1",
    "--batch_size", "1",
    "--eval_batch_size", "2",
    "--gradient_accumulation_steps", "4",
    "--seed", str(SEED),
]
if DEVICE == "cuda":
    lettuce_train_cmd.append("--fp16")

run_checked(lettuce_train_cmd)


### Train LookBack Lens classifier

LookBack Lens is different from LettuceDetect: it first extracts attention-based lookback-ratio features from a causal language model, then trains a logistic-regression classifier on those features. Because this is expensive, `QUICK_RUN` trains it on a small shuffled subset and evaluates it on the same small comparison subset used for LettuceDetect below. For the full experiment, set `QUICK_RUN = False`.


In [ ]:
# Train LookBack Lens classifier.
# We check for classifier.pkl, not just for the directory, because an empty/stale directory is not enough.

lookback_classifier = LOOKBACK_DIR / "classifier.pkl"
lookback_train_cmd = [
    sys.executable,
    "src/train_lookback_lens.py",
    "--dataset", str(TRAIN_DIR),
    "--output_dir", str(LOOKBACK_DIR),
    "--model", LOOKBACK_MODEL,
    "--max_length", str(LOOKBACK_MAX_LENGTH),
    "--device", DEVICE,
    "--seed", str(SEED),
]
if LOOKBACK_LIMIT is not None:
    lookback_train_cmd += ["--limit", str(LOOKBACK_LIMIT)]

if lookback_classifier.exists():
    print(f"Using existing LookBack Lens classifier: {lookback_classifier.resolve()}")
else:
    run_checked(lookback_train_cmd)

if not lookback_classifier.exists():
    raise FileNotFoundError(f"LookBack Lens training did not create {lookback_classifier}")


### Evaluation helper

The parser below extracts the span-level `TP`, `FP`, `FN`, precision, recall, and F1 printed by `src/eval_baselines.py`. If a baseline fails, the helper prints the real subprocess error first, which makes Colab debugging much easier.


In [ ]:
# Shared evaluator.

def run_eval(method: str, dataset_path: Path, **kwargs) -> dict:
    cmd = [
        sys.executable,
        "src/eval_baselines.py",
        "--dataset", str(dataset_path),
        "--method", method,
    ]
    if method == "lettucedetect":
        cmd += ["--lettuce_model", str(kwargs.get("lettuce_model", LETTUCE_BASE_MODEL)), "--device", DEVICE]
    if method == "lookback_lens":
        cmd += [
            "--lookback_classifier", str(kwargs.get("lookback_classifier", LOOKBACK_DIR)),
            "--lookback_model", str(kwargs.get("lookback_model", LOOKBACK_MODEL)),
            "--device", DEVICE,
        ]

    completed = run_checked(cmd)
    match = re.search(
        r"Method=(?P<method>\S+)\s+TP=(?P<tp>\d+) FP=(?P<fp>\d+) FN=(?P<fn>\d+) "
        r"P=(?P<precision>[0-9.]+) R=(?P<recall>[0-9.]+) F1=(?P<f1>[0-9.]+)",
        completed.stdout,
    )
    if not match:
        raise RuntimeError(f"Could not parse evaluator output:\n{completed.stdout}")
    row = match.groupdict()
    row["dataset"] = str(dataset_path)
    for key in ["tp", "fp", "fn"]:
        row[key] = int(row[key])
    for key in ["precision", "recall", "f1"]:
        row[key] = float(row[key])
    return row


### Quick comparison: LettuceDetect vs LookBack Lens

This is the direct comparison requested for the notebook. In `QUICK_RUN`, both models are evaluated on the same small mixed test subset so the cell finishes in a reasonable time. The full held-out test set contains 1944 examples and can be used by setting `QUICK_RUN = False` at the top of this section.


In [ ]:
# Direct LettuceDetect vs LookBack Lens comparison on the same data.

comparison_results = []
comparison_results.append(
    run_eval("lettucedetect", COMPARISON_DATASET, lettuce_model=LETTUCE_OUTPUT_DIR)
)
comparison_results.append(
    run_eval("lookback_lens", COMPARISON_DATASET, lookback_classifier=LOOKBACK_DIR, lookback_model=LOOKBACK_MODEL)
)

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df[["method", "dataset", "tp", "fp", "fn", "precision", "recall", "f1"]])

comparison_path = Path("outputs") / "lettuce_vs_lookback_metrics.json"
comparison_path.parent.mkdir(exist_ok=True)
comparison_path.write_text(json.dumps(comparison_results, indent=2), encoding="utf-8")
print(f"Saved Lettuce vs LookBack comparison to {comparison_path.resolve()}")


### Baseline table

For context, the final table also includes the simple `tool_overlap` heuristic. In quick mode this table uses the same comparison subset as the neural baseline comparison above.


In [ ]:
# Baseline table including the heuristic.

results = []
for method, kwargs in [
    ("tool_overlap", {}),
    ("lettucedetect", {"lettuce_model": LETTUCE_OUTPUT_DIR}),
    ("lookback_lens", {"lookback_classifier": LOOKBACK_DIR, "lookback_model": LOOKBACK_MODEL}),
]:
    results.append(run_eval(method, COMPARISON_DATASET, **kwargs))

results_df = pd.DataFrame(results)
display(results_df[["method", "dataset", "tp", "fp", "fn", "precision", "recall", "f1"]])

results_path = Path("outputs") / "assignment_metrics.json"
results_path.parent.mkdir(exist_ok=True)
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Saved metrics to {results_path.resolve()}")
